In [6]:
#!/usr/bin/env python3
"""
Temporary script to examine neural.ns6 header information
Fixed: Removed close() calls since BlackrockIO doesn't have a close() method
"""

import warnings
warnings.filterwarnings('ignore')

# Import Neo for BlackRock file reading
try:
    import neo
    from neo.io import BlackrockIO
    print("✅ Neo library imported successfully")
except ImportError:
    print("❌ Neo library not found. Please install with: pip install neo")
    print("Run: pip install neo")
    
# Path to neural.ns6 file (adjust as needed)
NEURAL_FILE = r"D:\Data\ScienceCorp\neural.ns6"

print(f"🔍 Examining NS6 Header Information")
print("=" * 50)
print(f"📁 File: {NEURAL_FILE}")
print()


✅ Neo library imported successfully
🔍 Examining NS6 Header Information
📁 File: D:\Data\ScienceCorp\neural.ns6



In [7]:
import os
from pathlib import Path
import pprint

# Check if file exists
if not Path(NEURAL_FILE).exists():
    print(f"❌ Neural file not found: {NEURAL_FILE}")
    print("Please update the NEURAL_FILE path in the cell above to match your file location")
else:
    print(f"✅ Neural file found: {NEURAL_FILE}")
    
    try:
        # Create Neo reader for the NS6 file
        print("\n📂 Creating BlackrockIO reader...")
        reader = BlackrockIO(filename=NEURAL_FILE)
        print("✅ Reader created successfully!")
        
        print("\n" + "="*60)
        print("📊 NS6 HEADER INFORMATION")
        print("="*60)
        
        # Method 1: Direct header access
        print("\n🔍 Method 1: reader.header")
        print("-" * 30)
        if hasattr(reader, 'header'):
            header = reader.header
            print("Header type:", type(header))
            print("Header contents:")
            pprint.pprint(header, depth=3, width=80)
        else:
            print("No direct header attribute found")
            
        # Method 2: Check for time-related attributes
        print("\n🔍 Method 2: Time-related attributes")
        print("-" * 30)
        
        time_attrs = ['datetime', 'rec_datetime', 'file_datetime', 'time_origin']
        for attr in time_attrs:
            if hasattr(reader, attr):
                value = getattr(reader, attr)
                print(f"  • {attr}: {value} (type: {type(value)})")
            else:
                print(f"  • {attr}: Not found")
                
        # Method 3: Read header method
        print("\n🔍 Method 3: _read_header() method")
        print("-" * 30)
        if hasattr(reader, '_read_header'):
            try:
                header_info = reader._read_header()
                print("Header info type:", type(header_info))
                print("Header info contents:")
                pprint.pprint(header_info, depth=3, width=80)
            except Exception as e:
                print(f"Error calling _read_header(): {e}")
        else:
            print("No _read_header method found")
            
        # Method 4: Check all attributes
        print("\n🔍 Method 4: All reader attributes")
        print("-" * 30)
        all_attrs = [attr for attr in dir(reader) if not attr.startswith('_')]
        interesting_attrs = [attr for attr in all_attrs if any(keyword in attr.lower() for keyword in ['time', 'date', 'header', 'info', 'meta'])]
        
        print("Interesting attributes:")
        for attr in interesting_attrs:
            try:
                value = getattr(reader, attr)
                # Don't call methods, just show what's available
                if callable(value):
                    print(f"  • {attr}: <method>")
                else:
                    print(f"  • {attr}: {value}")
            except Exception as e:
                print(f"  • {attr}: Error accessing - {e}")
                
        print(f"\nAll available attributes ({len(all_attrs)} total):")
        print(", ".join(all_attrs))
        
        # BlackrockIO doesn't have a close() method, so we just let it go out of scope
        print("\n✅ Reader processing completed successfully")
        
    except Exception as e:
        print(f"❌ Error reading NS6 file: {e}")
        import traceback
        traceback.print_exc()


✅ Neural file found: D:\Data\ScienceCorp\neural.ns6

📂 Creating BlackrockIO reader...
✅ Reader created successfully!

📊 NS6 HEADER INFORMATION

🔍 Method 1: reader.header
------------------------------
Header type: <class 'dict'>
Header contents:
{'event_channels': array([], dtype=[('name', '<U64'), ('id', '<U64'), ('type', 'S5')]),
 'nb_block': 1,
 'nb_segment': [1],
 'signal_buffers': array([('nsx6', '6')], dtype=[('name', '<U64'), ('id', '<U64')]),
 'signal_channels': array([('Hub1-chan1', '1', 30000., 'int16', 'uV', 0.25, 0., '6', '6'),
       ('Hub1-chan2', '2', 30000., 'int16', 'uV', 0.25, 0., '6', '6'),
       ('Hub1-chan3', '3', 30000., 'int16', 'uV', 0.25, 0., '6', '6'),
       ('Hub1-chan4', '4', 30000., 'int16', 'uV', 0.25, 0., '6', '6'),
       ('Hub1-chan5', '5', 30000., 'int16', 'uV', 0.25, 0., '6', '6'),
       ('Hub1-chan6', '6', 30000., 'int16', 'uV', 0.25, 0., '6', '6'),
       ('Hub1-chan7', '7', 30000., 'int16', 'uV', 0.25, 0., '6', '6'),
       ('Hub1-chan8', '8', 3

In [8]:
# Additional exploration of NS6 file structure
print("\n" + "="*60)
print("📋 ADDITIONAL FILE STRUCTURE INFORMATION")
print("="*60)

if Path(NEURAL_FILE).exists():
    try:
        # File size information
        file_size = Path(NEURAL_FILE).stat().st_size
        print(f"\n📏 File Size: {file_size:,} bytes ({file_size/1024**3:.2f} GB)")
        
        # Re-open reader to get block/segment information
        reader = BlackrockIO(filename=NEURAL_FILE)
        
        # Get file info without loading full data
        print("\n📊 Block and Segment Information:")
        print("-" * 40)
        
        # Try to get basic info about the data structure
        try:
            # Get minimal block info
            block = reader.read_block(lazy=True)  # Lazy loading to avoid memory issues
            print(f"  • Block type: {type(block)}")
            print(f"  • Number of segments: {len(block.segments)}")
            
            if len(block.segments) > 0:
                segment = block.segments[0]
                print(f"  • Segment type: {type(segment)}")
                print(f"  • Number of analog signals: {len(segment.analogsignals)}")
                
                if len(segment.analogsignals) > 0:
                    analog_signal = segment.analogsignals[0]
                    print(f"  • Analog signal type: {type(analog_signal)}")
                    print(f"  • Signal shape: {analog_signal.shape}")
                    print(f"  • Sampling rate: {analog_signal.sampling_rate}")
                    print(f"  • Signal dtype: {analog_signal.dtype}")
                    
                    # Check for time information in the signal
                    if hasattr(analog_signal, 't_start'):
                        print(f"  • Signal t_start: {analog_signal.t_start}")
                    if hasattr(analog_signal, 't_stop'):
                        print(f"  • Signal t_stop: {analog_signal.t_stop}")
                        
                    # Check for annotations
                    if hasattr(analog_signal, 'annotations'):
                        print(f"  • Signal annotations: {analog_signal.annotations}")
                        
            # Block-level annotations
            if hasattr(block, 'annotations'):
                print(f"\n📝 Block Annotations:")
                print("-" * 20)
                for key, value in block.annotations.items():
                    print(f"  • {key}: {value}")
                    
        except Exception as e:
            print(f"  Error getting block info: {e}")
            
        # BlackrockIO doesn't have a close() method, reader will be cleaned up automatically
        
        print("\n✅ Header exploration complete!")
        print("\nTo use this file in your analysis, you can:")
        print("1. Load it with: reader = neo.BlackrockIO(filename='path/to/neural.ns6')")
        print("2. Access header with: header = reader.header")
        print("3. Get time origin from header for alignment with behavioral data")
        
    except Exception as e:
        print(f"❌ Error exploring file structure: {e}")
        import traceback
        traceback.print_exc()
else:
    print("❌ File not found - cannot explore structure")
    print("Please update the NEURAL_FILE path in the first cell")



📋 ADDITIONAL FILE STRUCTURE INFORMATION

📏 File Size: 9,232,017,855 bytes (8.60 GB)

📊 Block and Segment Information:
----------------------------------------
  • Block type: <class 'neo.core.block.Block'>
  • Number of segments: 1
  • Segment type: <class 'neo.core.segment.Segment'>
  • Number of analog signals: 1
  • Analog signal type: <class 'neo.io.proxyobjects.AnalogSignalProxy'>
  • Signal shape: (45034200, 96)
  • Sampling rate: 30000.0 Hz
  • Signal dtype: int16
  • Signal t_start: 1739738583.3854105 s
  • Signal t_stop: 1739740084.5254107 s
  • Signal annotations: {'stream_id': '6', 'nsx': 6}

📝 Block Annotations:
--------------------
  • avail_file_set: ['ns6']
  • avail_nsx: [6]
  • avail_nev: False
  • rec_pauses: False

✅ Header exploration complete!

To use this file in your analysis, you can:
1. Load it with: reader = neo.BlackrockIO(filename='path/to/neural.ns6')
2. Access header with: header = reader.header
3. Get time origin from header for alignment with behavioral

In [9]:
# DEEPER EXPLORATION: Looking for timestamp information in all possible locations
print("\n" + "="*60)
print("🔍 DEEP DIVE: TIMESTAMP HUNTING IN NS6 FILE")
print("="*60)

if Path(NEURAL_FILE).exists():
    try:
        reader = BlackrockIO(filename=NEURAL_FILE)
        
        print("\n📅 SEARCH 1: Raw file header exploration")
        print("-" * 50)
        
        # Try to access raw header information
        if hasattr(reader, 'header'):
            header = reader.header
            print("Full header contents:")
            for key, value in header.items():
                print(f"  • {key}: {value}")
                # Look for anything that might contain time/date info
                if isinstance(value, dict):
                    print(f"    └─ (nested dict with {len(value)} items)")
                    for sub_key, sub_value in value.items():
                        print(f"       • {sub_key}: {sub_value}")
        
        print("\n📅 SEARCH 2: Block-level information")
        print("-" * 50)
        
        # Load block and explore all its attributes
        block = reader.read_block(lazy=True)
        
        # Check ALL block attributes for potential time information
        block_attrs = dir(block)
        time_related_attrs = [attr for attr in block_attrs if any(keyword in attr.lower() for keyword in ['time', 'date', 'origin', 'start', 'stamp', 'created', 'modified'])]
        
        print("Time-related block attributes:")
        for attr in time_related_attrs:
            if not attr.startswith('_'):
                try:
                    value = getattr(block, attr)
                    print(f"  • block.{attr}: {value} (type: {type(value)})")
                except Exception as e:
                    print(f"  • block.{attr}: Error accessing - {e}")
        
        print("\n📅 SEARCH 3: Block annotations deep dive")
        print("-" * 50)
        
        if hasattr(block, 'annotations') and block.annotations:
            print("Block annotations:")
            for key, value in block.annotations.items():
                print(f"  • {key}: {value}")
                # Look for nested time information
                if isinstance(value, dict):
                    print(f"    └─ (nested dict):")
                    for sub_key, sub_value in value.items():
                        print(f"       • {sub_key}: {sub_value}")
        else:
            print("No block annotations found")
        
        print("\n📅 SEARCH 4: Segment-level information")
        print("-" * 50)
        
        if len(block.segments) > 0:
            segment = block.segments[0]
            
            # Check segment attributes
            segment_attrs = dir(segment)
            time_related_attrs = [attr for attr in segment_attrs if any(keyword in attr.lower() for keyword in ['time', 'date', 'origin', 'start', 'stamp', 'created', 'modified'])]
            
            print("Time-related segment attributes:")
            for attr in time_related_attrs:
                if not attr.startswith('_'):
                    try:
                        value = getattr(segment, attr)
                        print(f"  • segment.{attr}: {value} (type: {type(value)})")
                    except Exception as e:
                        print(f"  • segment.{attr}: Error accessing - {e}")
            
            # Check segment annotations
            if hasattr(segment, 'annotations') and segment.annotations:
                print("\nSegment annotations:")
                for key, value in segment.annotations.items():
                    print(f"  • {key}: {value}")
        
        print("\n📅 SEARCH 5: Analog signal information")
        print("-" * 50)
        
        if len(block.segments) > 0 and len(block.segments[0].analogsignals) > 0:
            analog_signal = block.segments[0].analogsignals[0]
            
            # Check analog signal attributes
            signal_attrs = dir(analog_signal)
            time_related_attrs = [attr for attr in signal_attrs if any(keyword in attr.lower() for keyword in ['time', 'date', 'origin', 'start', 'stamp', 'created', 'modified'])]
            
            print("Time-related analog signal attributes:")
            for attr in time_related_attrs:
                if not attr.startswith('_'):
                    try:
                        value = getattr(analog_signal, attr)
                        print(f"  • analog_signal.{attr}: {value} (type: {type(value)})")
                    except Exception as e:
                        print(f"  • analog_signal.{attr}: Error accessing - {e}")
            
            # Check analog signal annotations
            if hasattr(analog_signal, 'annotations') and analog_signal.annotations:
                print("\nAnalog signal annotations:")
                for key, value in analog_signal.annotations.items():
                    print(f"  • {key}: {value}")
        
        # BlackrockIO doesn't have a close() method, reader will be cleaned up automatically
        
    except Exception as e:
        print(f"❌ Error in deep exploration: {e}")
        import traceback
        traceback.print_exc()
else:
    print("❌ File not found")



🔍 DEEP DIVE: TIMESTAMP HUNTING IN NS6 FILE

📅 SEARCH 1: Raw file header exploration
--------------------------------------------------
Full header contents:
  • nb_block: 1
  • nb_segment: [1]
  • signal_buffers: [('nsx6', '6')]
  • signal_streams: [('nsx6', '6', '6')]
  • signal_channels: [('Hub1-chan1', '1', 30000., 'int16', 'uV', 0.25, 0., '6', '6')
 ('Hub1-chan2', '2', 30000., 'int16', 'uV', 0.25, 0., '6', '6')
 ('Hub1-chan3', '3', 30000., 'int16', 'uV', 0.25, 0., '6', '6')
 ('Hub1-chan4', '4', 30000., 'int16', 'uV', 0.25, 0., '6', '6')
 ('Hub1-chan5', '5', 30000., 'int16', 'uV', 0.25, 0., '6', '6')
 ('Hub1-chan6', '6', 30000., 'int16', 'uV', 0.25, 0., '6', '6')
 ('Hub1-chan7', '7', 30000., 'int16', 'uV', 0.25, 0., '6', '6')
 ('Hub1-chan8', '8', 30000., 'int16', 'uV', 0.25, 0., '6', '6')
 ('Hub1-chan9', '9', 30000., 'int16', 'uV', 0.25, 0., '6', '6')
 ('Hub1-chan10', '10', 30000., 'int16', 'uV', 0.25, 0., '6', '6')
 ('Hub1-chan11', '11', 30000., 'int16', 'uV', 0.25, 0., '6', '6')


In [10]:
# FINAL EXPLORATION: Looking at raw binary header data if needed
print("\n" + "="*60)
print("🔍 FINAL SEARCH: RAW BINARY HEADER DATA")
print("="*60)

if Path(NEURAL_FILE).exists():
    try:
        # Sometimes the time origin is stored in the raw binary header
        # Let's check if we can access it directly
        
        print("\n📅 SEARCH 6: Raw binary header parsing")
        print("-" * 50)
        
        # Open the file in binary mode to examine raw header
        with open(NEURAL_FILE, 'rb') as f:
            # Read first 1024 bytes to look for header information
            raw_header = f.read(1024)
            
            print(f"First 1024 bytes of file (hex):")
            hex_data = raw_header.hex()
            # Print in groups of 32 characters (16 bytes) for readability
            for i in range(0, min(len(hex_data), 512), 32):  # Show first 256 bytes
                print(f"  {i//2:04x}: {hex_data[i:i+32]}")
            
            print(f"\nFirst 1024 bytes (ASCII representation):")
            ascii_data = ''.join(chr(b) if 32 <= b <= 126 else '.' for b in raw_header)
            # Print in groups of 64 characters for readability
            for i in range(0, min(len(ascii_data), 512), 64):  # Show first 512 chars
                print(f"  {i:04x}: {ascii_data[i:i+64]}")
        
        print("\n📅 SEARCH 7: Neo internal header methods")
        print("-" * 50)
        
        # Try to access Neo's internal header parsing methods
        reader = BlackrockIO(filename=NEURAL_FILE)
        
        # Look for private methods that might contain header info
        private_methods = [attr for attr in dir(reader) if attr.startswith('_') and 'header' in attr.lower()]
        print("Private header-related methods:")
        for method in private_methods:
            print(f"  • {method}")
        
        # Try some specific methods that might exist
        header_methods = ['_read_header', '_parse_header', '_get_header_info']
        for method_name in header_methods:
            if hasattr(reader, method_name):
                try:
                    print(f"\nTrying {method_name}():")
                    method = getattr(reader, method_name)
                    if callable(method):
                        result = method()
                        print(f"  Result type: {type(result)}")
                        if isinstance(result, dict):
                            for key, value in result.items():
                                print(f"  • {key}: {value}")
                        else:
                            print(f"  Result: {result}")
                    else:
                        print(f"  {method_name} is not callable")
                except Exception as e:
                    print(f"  Error calling {method_name}: {e}")
        
        # BlackrockIO doesn't have a close() method, reader will be cleaned up automatically
        
        print("\n📅 SUMMARY: What we found")
        print("-" * 50)
        print("If no time origin was found above, it suggests:")
        print("1. The NS6 file may not contain embedded timestamp information")
        print("2. The time origin might be stored in a separate .nev file")
        print("3. The time origin might need to be provided externally")
        print("4. The existing code uses a hardcoded fallback time origin")
        print("\nCheck your project's existing code - it may already handle this case!")
        
    except Exception as e:
        print(f"❌ Error in final exploration: {e}")
        import traceback
        traceback.print_exc()
else:
    print("❌ File not found")



🔍 FINAL SEARCH: RAW BINARY HEADER DATA

📅 SEARCH 6: Raw binary header parsing
--------------------------------------------------
First 1024 bytes of file (hex):
  0000: 4252534d504752500300fa1900007261
  0010: 77000000000000000000000000000000
  0020: 00000000000000000000000000000000
  0030: 00000000000000000000000000000000
  0040: 00000000000000000000000000000000
  0050: 00000000000000000000000000000000
  0060: 00000000000000000000000000000000
  0070: 00000000000000000000000000000000
  0080: 00000000000000000000000000000000
  0090: 00000000000000000000000000000000
  00a0: 00000000000000000000000000000000
  00b0: 00000000000000000000000000000000
  00c0: 00000000000000000000000000000000
  00d0: 00000000000000000000000000000000
  00e0: 00000000000000000000000000000000
  00f0: 00000000000000000000000000000000

First 1024 bytes (ASCII representation):
  0000: BRSMPGRP......raw...............................................
  0040: ...........................................................

In [11]:
# BLACKROCK NSX HEADER PARSER - Following the official specification
print("\n" + "="*60)
print("🔍 BLACKROCK NSX HEADER PARSER (Official Specification)")
print("="*60)

import struct
from datetime import datetime, timezone

def parse_nsx_header(file_path):
    """
    Parse BlackRock NSX file header according to official specification.
    
    Returns a dictionary with parsed header fields including the Time Origin.
    """
    
    if not Path(file_path).exists():
        return None
    
    try:
        with open(file_path, 'rb') as f:
            header = {}
            
            # Read the basic header according to specification
            print("📋 Parsing Basic Header (Official NSX Specification)")
            print("-" * 50)
            
            # File Type ID (8 bytes) - char array
            file_type_id = f.read(8)
            header['file_type_id'] = file_type_id.decode('ascii', errors='ignore').rstrip('\x00')
            print(f"  • File Type ID: '{header['file_type_id']}'")
            
            # File Spec (2 bytes) - 2 x unsigned char
            file_spec = struct.unpack('<BB', f.read(2))
            header['file_spec'] = f"{file_spec[0]}.{file_spec[1]}"
            print(f"  • File Spec: {header['file_spec']}")
            
            # Bytes in Headers (4 bytes) - unsigned int-32
            bytes_in_headers = struct.unpack('<I', f.read(4))[0]
            header['bytes_in_headers'] = bytes_in_headers
            print(f"  • Bytes in Headers: {bytes_in_headers}")
            
            # Label (16 bytes) - char array
            label = f.read(16)
            header['label'] = label.decode('ascii', errors='ignore').rstrip('\x00')
            print(f"  • Label: '{header['label']}'")
            
            # Comment (256 bytes) - char array
            comment = f.read(256)
            header['comment'] = comment.decode('ascii', errors='ignore').rstrip('\x00')
            print(f"  • Comment: '{header['comment']}'")
            
            # Period (4 bytes) - unsigned int-32
            period = struct.unpack('<I', f.read(4))[0]
            header['period'] = period
            sampling_rate = 30000 / period if period > 0 else 0
            print(f"  • Period: {period} (Sampling Rate: {sampling_rate:.1f} Hz)")
            
            # Time Resolution (4 bytes) - unsigned int-32
            time_resolution = struct.unpack('<I', f.read(4))[0]
            header['time_resolution'] = time_resolution
            print(f"  • Time Resolution: {time_resolution} Hz")
            
            # ⭐ TIME ORIGIN (16 bytes) - Windows SYSTEM TIME structure ⭐
            print(f"\n🎯 TIME ORIGIN (Windows SYSTEM TIME structure):")
            print("-" * 30)
            
            # Read 16 bytes and parse as 8 unsigned int-16 values
            time_origin_bytes = f.read(16)
            time_values = struct.unpack('<8H', time_origin_bytes)  # 8 unsigned int-16, little-endian
            
            year, month, day_of_week, day, hour, minute, second, millisecond = time_values
            
            print(f"  • Raw values: {time_values}")
            print(f"  • Year: {year}")
            print(f"  • Month: {month}")
            print(f"  • DayOfWeek: {day_of_week}")
            print(f"  • Day: {day}")
            print(f"  • Hour: {hour}")
            print(f"  • Minute: {minute}")
            print(f"  • Second: {second}")
            print(f"  • Millisecond: {millisecond}")
            
            # Convert to Python datetime (UTC)
            if year > 0 and month > 0 and day > 0:
                time_origin = datetime(year, month, day, hour, minute, second, 
                                     millisecond * 1000, timezone.utc)
                header['time_origin'] = time_origin
                print(f"\n  🎯 TIME ORIGIN: {time_origin}")
                print(f"  📅 Formatted: {time_origin.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]} UTC")
            else:
                print(f"  ⚠️  Invalid time values - cannot construct datetime")
                header['time_origin'] = None
            
            # Channel Count (4 bytes) - unsigned int-32
            channel_count = struct.unpack('<I', f.read(4))[0]
            header['channel_count'] = channel_count
            print(f"\n  • Channel Count: {channel_count}")
            
            return header
            
    except Exception as e:
        print(f"❌ Error parsing NSX header: {e}")
        import traceback
        traceback.print_exc()
        return None

# Parse the header
if Path(NEURAL_FILE).exists():
    print(f"📁 Parsing: {NEURAL_FILE}")
    header_info = parse_nsx_header(NEURAL_FILE)
    
    if header_info:
        print(f"\n✅ Header parsed successfully!")
        print(f"\n📊 SUMMARY:")
        print(f"  • File Type: {header_info.get('file_type_id', 'Unknown')}")
        print(f"  • File Spec: {header_info.get('file_spec', 'Unknown')}")
        print(f"  • Sampling Rate: {30000 / header_info.get('period', 1):.1f} Hz")
        print(f"  • Channels: {header_info.get('channel_count', 'Unknown')}")
        print(f"  • Time Origin: {header_info.get('time_origin', 'Not found')}")
        
        # Compare with expected time origin from your code
        expected_time = datetime(2025, 3, 25, 9, 22, 53, tzinfo=timezone.utc)
        if header_info.get('time_origin'):
            actual_time = header_info['time_origin']
            time_diff = abs((actual_time - expected_time).total_seconds())
            print(f"\n🔍 Comparison with expected time origin:")
            print(f"  • Expected: {expected_time}")
            print(f"  • Actual:   {actual_time}")
            print(f"  • Difference: {time_diff:.3f} seconds")
            if time_diff < 1.0:
                print(f"  ✅ Times match closely!")
            else:
                print(f"  ⚠️  Times differ by {time_diff:.1f} seconds")
    else:
        print(f"❌ Failed to parse header")
else:
    print(f"❌ File not found: {NEURAL_FILE}")



🔍 BLACKROCK NSX HEADER PARSER (Official Specification)
📁 Parsing: D:\Data\ScienceCorp\neural.ns6
📋 Parsing Basic Header (Official NSX Specification)
--------------------------------------------------
  • File Type ID: 'BRSMPGRP'
  • File Spec: 3.0
  • Bytes in Headers: 6650
  • Label: 'raw'
  • Comment: ''
  • Period: 1 (Sampling Rate: 30000.0 Hz)
  • Time Resolution: 1000000000 Hz

🎯 TIME ORIGIN (Windows SYSTEM TIME structure):
------------------------------
  • Raw values: (2025, 3, 2, 25, 21, 22, 53, 360)
  • Year: 2025
  • Month: 3
  • DayOfWeek: 2
  • Day: 25
  • Hour: 21
  • Minute: 22
  • Second: 53
  • Millisecond: 360

  🎯 TIME ORIGIN: 2025-03-25 21:22:53.360000+00:00
  📅 Formatted: 2025-03-25 21:22:53.360 UTC

  • Channel Count: 96

✅ Header parsed successfully!

📊 SUMMARY:
  • File Type: BRSMPGRP
  • File Spec: 3.0
  • Sampling Rate: 30000.0 Hz
  • Channels: 96
  • Time Origin: 2025-03-25 21:22:53.360000+00:00

🔍 Comparison with expected time origin:
  • Expected: 2025-03-2